# Fig. A1：Barillas–Shanken 解析贝叶斯比较

在主数据、因子与检验 notebook 后运行。本文件不下载或保存原始PDF，不修改03。输入是已有月因子 `f_ch3`、`f_ff3`；市场必须相同，取两模型全部因子的共同完整月份，报告实际199月等覆盖，而不填补成论文204月。

此次已找到原始最终稿：[Barillas & Shanken, Comparing Asset Pricing Models](https://fnce.wharton.upenn.edu/wp-content/uploads/2017/10/BarillasShanken-9.21.17.pdf)。可计算公式来自式(7)、(9)–(11)、(19)，Appendix A.1/A.2和B.1/B.2（PDF第12、15–16、26–27、68–70页）。[Stambaugh & Yuan的作者机构存档](https://repository.upenn.edu/bitstreams/40573e11-6381-498d-abe0-76ebe90b8401/download)及本地《Size and value in China》A7使用等模型先验、Sharpe乘数横轴与该方法。

以下实现公开的解析方法。RESSET构造/口径与论文WIND、严格交易资格导致的月份缺口、无风险收益精度仍是数据限制；输出是本地因子模型比较，不声称完全重现原图曲线。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import multigammaln, logsumexp
from scipy.integrate import quad
from scipy.stats import invgamma
from IPython.display import display

## 从解析积分到计算

设样本T个月，N个左侧资产、K个右侧因子（不含截距），F为右侧原始收益矩阵。S为含截距OLS残差交叉乘积，S_R为零截距OLS残差交叉乘积；协方差使用MLE S/T，不能改成残差自由度版本。

令 $a=(1+Sh(F)^2)/T$，$W/T=\hat\alpha'(S/T)^{-1}\hat\alpha/(1+Sh(F)^2)$，先验为 $\alpha|\Sigma\sim N(0,k\Sigma)$。则

$$\log Q=-\frac{T-K}{2}\log\left[1+\frac{a}{a+k}\frac WT\right]-\frac N2\log(1+k/a).$$

$$\log C=-\frac{N(T-K)}2\log\pi+\log\Gamma_N((T-K)/2)-\frac N2\log|F'F|.$$

$$\log ML_U=\log C-\frac{T-K}{2}\log|S|+\log Q,\qquad
\log ML_R=\log C-\frac{T-K}{2}\log|S_R|.$$

每个模型 $M$: $ML(M)=ML_U(f_M|Mkt)ML_R(f_{excluded}|Mkt,f_M)$；共同测试资产项相消。这里两模型都含市场和两个非市场因子；候选并集为五因子。按原文第27页的全候选集校准，$k=(m^2-1)Sh(Mkt)^2/4$，m是横轴乘数。每增加一个非市场候选因子的预期平方Sharpe增量相同，不把分母错写成单个模型的两个因子。

注意：原文Fig.A1的横轴文字使用“model's factors”；代码采用Barillas–Shanken明确的全候选集规则，并保留校准参数信息。是否与作者未公开的原图计算脚本每项一致仍待作者代码核对。m=1是k=0的极限，各纳入因子alpha也被压为0，允许直接用解析极限，不以任意epsilon改先验。

In [ ]:
def get_positive_logdet(matrix, label):
    sign, value = np.linalg.slogdet(matrix)
    if sign <= 0 or not np.isfinite(value):
        raise ValueError(label + '必须正定且非奇异')
    return float(value)


def get_bs_regression_statistics(returns, factors):
    y = np.asarray(returns, dtype=float)
    f = np.asarray(factors, dtype=float)
    if y.ndim == 1:
        y = y[:, None]
    if f.ndim == 1:
        f = f[:, None]
    if y.ndim != 2 or f.ndim != 2 or len(y) != len(f):
        raise ValueError('输入应为同T的二维矩阵')
    if not np.isfinite(y).all() or not np.isfinite(f).all():
        raise ValueError('回归只接受共同完整有限样本')
    t, n = y.shape
    k = f.shape[1]
    if n < 1 or k < 1 or t <= n+k:
        raise ValueError('贝叶斯样本不足或因子为空')
    design = np.column_stack([np.ones(t), f])
    if np.linalg.matrix_rank(design) < k+1:
        raise ValueError('解释因子与截距线性相关')
    coefficient = np.linalg.lstsq(design, y, rcond=None)[0]
    residual = y - design @ coefficient
    restricted_coefficient = np.linalg.lstsq(f, y, rcond=None)[0]
    restricted_residual = y - f @ restricted_coefficient
    s = residual.T @ residual
    s_restricted = restricted_residual.T @ restricted_residual
    centered_factors = f - f.mean(axis=0)
    factor_covariance = centered_factors.T @ centered_factors / t
    get_positive_logdet(factor_covariance, '因子MLE协方差')
    sharpe_squared = float(f.mean(axis=0) @ np.linalg.solve(factor_covariance, f.mean(axis=0)))
    alpha = coefficient[0]
    a = (1 + sharpe_squared) / t
    w_over_t = float(alpha @ np.linalg.solve(s / t, alpha) / (1 + sharpe_squared))
    log_constant = (-n * (t-k) / 2 * np.log(np.pi) + multigammaln((t-k)/2, n)
                    -n/2 * get_positive_logdet(f.T @ f, 'F转置F'))
    return {'t': t, 'n': n, 'k': k, 'a': a, 'w_over_t': w_over_t,
            'sharpe_squared': sharpe_squared, 'log_constant': float(log_constant),
            'logdet_s': get_positive_logdet(s, 'S'),
            'logdet_s_restricted': get_positive_logdet(s_restricted, 'S_R'),
            'alpha': alpha, 's': s}


def get_bs_log_marginal(statistics, prior_variance=0.0, restricted=False):
    if prior_variance < 0 or not np.isfinite(prior_variance):
        raise ValueError('先验比例k必须有限非负')
    exponent = (statistics['t'] - statistics['k']) / 2
    if restricted:
        return statistics['log_constant'] - exponent * statistics['logdet_s_restricted']
    a = statistics['a']
    log_q = (-exponent * np.log1p(a / (a + prior_variance) * statistics['w_over_t'])
             -statistics['n']/2 * np.log1p(prior_variance/a))
    return statistics['log_constant'] - exponent * statistics['logdet_s'] + log_q


def get_ch3_ff3_bayesian_comparison(ch3, ff3, multipliers=None):
    names_ch3 = ['market_factor', 'size_factor_smb', 'value_factor_vmg']
    names_ff3 = ['market_factor', 'ff_size_factor', 'ff_value_factor']
    for frame, names in [(ch3, names_ch3), (ff3, names_ff3)]:
        if not frame.index.is_unique or any(name not in frame for name in names):
            raise ValueError('因子列缺失或月份重复')
        if not isinstance(frame.index, pd.PeriodIndex) or frame.index.freqstr != 'M':
            raise ValueError('因子行轴必须是唯一Period[M]')
    if multipliers is None:
        multipliers = np.linspace(1.0, 2.0, 201)
    multipliers = np.asarray(multipliers, dtype=float)
    if not np.isfinite(multipliers).all() or (multipliers < 1).any():
        raise ValueError('Sharpe乘数必须有限且不小于1')
    common = ch3[names_ch3].join(ff3[names_ff3], how='inner', lsuffix='_ch3', rsuffix='_ff3')
    common = common.loc[pd.Period('2000-01',freq='M'):pd.Period('2016-12',freq='M')]
    common = common.replace([np.inf,-np.inf], np.nan).dropna().sort_index()
    if not np.allclose(common['market_factor_ch3'], common['market_factor_ff3'], rtol=1e-10, atol=1e-12):
        raise ValueError('两模型市场因子不相同')
    market = common[['market_factor_ch3']]
    if len(market) < 8 or market.var(ddof=0).iloc[0] <= 0:
        raise ValueError('市场样本不足或无变化')
    market_sharpe_squared = float(market.mean().iloc[0] ** 2 / market.var(ddof=0).iloc[0])
    if market_sharpe_squared == 0:
        raise ValueError('市场均值为0，Sharpe乘数校准不可识别')
    included = {'ch3': common[['size_factor_smb', 'value_factor_vmg']],
                'ff3': common[['ff_size_factor', 'ff_value_factor']]}
    statistics = {}
    for model, other in [('ch3', 'ff3'), ('ff3', 'ch3')]:
        statistics[model] = {
            'unrestricted': get_bs_regression_statistics(included[model], market),
            'restricted': get_bs_regression_statistics(included[other], pd.concat([market,included[model]],axis=1))}
    rows = []
    for multiplier in multipliers:
        prior_variance = (multiplier ** 2 - 1) * market_sharpe_squared / 4
        log_ml = {}
        for model in ['ch3','ff3']:
            log_ml[model] = (get_bs_log_marginal(statistics[model]['unrestricted'], prior_variance)
                             +get_bs_log_marginal(statistics[model]['restricted'], restricted=True))
        normalizer = logsumexp([log_ml['ch3'], log_ml['ff3']])
        rows.append({'multiplier': multiplier, 'prior_k': prior_variance,
                     'ch3_log_ml': log_ml['ch3'], 'ff3_log_ml': log_ml['ff3'],
                     'ch3_probability': np.exp(log_ml['ch3']-normalizer),
                     'ff3_probability': np.exp(log_ml['ff3']-normalizer)})
    coverage = pd.DataFrame([{'months': len(common), 'paper_months': 204, 'start': common.index.min(), 'end': common.index.max(),
                             'missing_months': list(pd.period_range('2000-01','2016-12',freq='M').difference(common.index)),
                             'market_sharpe_monthly_abs': np.sqrt(market_sharpe_squared), 'candidate_nonmarket_factors': 4,
                             'model_prior_ch3': 0.5, 'model_prior_ff3': 0.5,
                             'status': '本地因子解析比较；全候选集先验校准，非原204月精确复制'}])
    return {'posterior': pd.DataFrame(rows), 'coverage': coverage, 'statistics': statistics}

## 行为与积分校验（人工样本）

检查k=0时非约束解析ML等于零截距ML；直接数值积分逆Gamma残差方差验证Q；模型对调后概率对调；概率和为1及非市场因子换列顺序不影响结果。人工样本不进入实证结果。

In [ ]:
def check_bayesian_analytic_calculation():
    rng = np.random.default_rng(76)
    t = 120
    market = rng.normal(0.006,0.04,t)
    assets = np.column_stack([0.005 + 0.3*market + rng.normal(0,0.03,t),
                              -0.002 + 0.6*market + rng.normal(0,0.05,t)])
    statistics = get_bs_regression_statistics(assets, market)
    assert np.isclose(get_bs_log_marginal(statistics, 0), get_bs_log_marginal(statistics, restricted=True), atol=1e-9)
    univariate = get_bs_regression_statistics(assets[:,0], market)
    prior = 0.003
    a = univariate['a']
    alpha = univariate['alpha'][0]
    shape = (univariate['t']-univariate['k'])/2
    scale = univariate['s'][0,0]/2
    # 先对正态alpha积分，再对逆Gamma方差作独立数值积分。
    integral, error = quad(lambda variance: np.exp(-alpha**2/(2*(a+prior)*variance))
                           * invgamma.pdf(variance, a=shape, scale=scale), 0, np.inf, epsabs=1e-10)
    integral /= np.sqrt(1+prior/a)
    analytic_q = np.exp(-shape*np.log1p(a/(a+prior)*univariate['w_over_t'])-0.5*np.log1p(prior/a))
    assert error < 1e-7 and np.isclose(integral, analytic_q, rtol=1e-7)
    months = pd.period_range('2000-01',periods=t,freq='M')
    other = np.column_stack([0.002+0.2*market+rng.normal(0,0.04,t),
                             0.003+0.1*market+rng.normal(0,0.05,t)])
    ch3 = pd.DataFrame({'market_factor': market, 'size_factor_smb': assets[:,0], 'value_factor_vmg': assets[:,1]},index=months)
    ff3 = pd.DataFrame({'market_factor': market, 'ff_size_factor': other[:,0], 'ff_value_factor': other[:,1]},index=months)
    first = get_ch3_ff3_bayesian_comparison(ch3,ff3,[1,1.05,1.5,2])['posterior']
    second = get_ch3_ff3_bayesian_comparison(ff3.rename(columns={'ff_size_factor':'size_factor_smb','ff_value_factor':'value_factor_vmg'}),
                                           ch3.rename(columns={'size_factor_smb':'ff_size_factor','value_factor_vmg':'ff_value_factor'}),[1,1.05,1.5,2])['posterior']
    assert np.allclose(first['ch3_probability']+first['ff3_probability'],1)
    assert np.allclose(first['ch3_probability'],second['ff3_probability'])
    reordered = get_ch3_ff3_bayesian_comparison(ch3.rename(columns={'size_factor_smb':'value_factor_vmg','value_factor_vmg':'size_factor_smb'}),ff3,[1,1.05,1.5,2])['posterior']
    assert np.allclose(first['ch3_probability'],reordered['ch3_probability'])
    return pd.DataFrame([{'check':'k0解析极限=零截距ML','passed':True}, {'check':'Q独立数值积分','passed':True},
                         {'check':'模型对调/归一化/因子次序','passed':True}])


bayesian_behaviour_checks = check_bayesian_analytic_calculation()
display(bayesian_behaviour_checks)

In [ ]:
if 'f_ch3' in globals() and 'f_ff3' in globals():
    try:
        bayesian_comparison = get_ch3_ff3_bayesian_comparison(f_ch3,f_ff3)
        display(bayesian_comparison['coverage'])
        selected = bayesian_comparison['posterior'][bayesian_comparison['posterior']['multiplier'].isin([1,1.05,1.1,1.25,1.5,2])]
        display(selected)
        ax = bayesian_comparison['posterior'].plot(x='multiplier',y=['ch3_probability','ff3_probability'],figsize=(9,4))
        ax.set(xlabel='Prior Sharpe multiplier (all candidate factors)',ylabel='Posterior probability',ylim=(0,1),
               title='CH3 versus FF3: analytic Bayesian comparison, local data')
        plt.tight_layout()
        display(ax.figure)
        plt.close(ax.figure)
        if 'extension_results' in globals():
            extension_results['fig_a1'] = bayesian_comparison
            extension_coverage_rows = [row for row in extension_coverage_rows if row['target']!='fig_a1']
            extension_coverage_rows.append({'target':'fig_a1','status':'本地因子解析比较已计算','detail':'BS全候选因子先验校准；实际共同月份见07；非原图204月精确复制'})
            extension_coverage = pd.DataFrame(extension_coverage_rows)
    except (ValueError, np.linalg.LinAlgError) as error:
        bayesian_comparison = {'status':'输入或解析计算未满足','detail':str(error)}
        display(bayesian_comparison)
else:
    bayesian_comparison = {'status':'缺因子矩阵','detail':'请先运行主数据与因子notebook'}
    display(bayesian_comparison)